# 🔢 MNIST Handwritten Digit Classification
## Deep Learning Project - Complete Implementation
---
**Is notebook mein yeh sab cover hoga:**
- ✅ Dataset Load & Preprocessing
- ✅ Activation Functions Comparison
- ✅ Neural Network Build & Train
- ✅ Overfitting Analysis + Dropout
- ✅ Confusion Matrix & Visualizations
- ✅ Expected Accuracy: ~98-99%

## 📦 Step 1: Libraries Import Karo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

# Reproducibility ke liye seed set karo
np.random.seed(42)
tf.random.set_seed(42)

print('✅ TensorFlow Version:', tf.__version__)
print('✅ GPU Available:', len(tf.config.list_physical_devices("GPU")) > 0)
print('✅ Sab libraries successfully import ho gayi!')

## 📊 Step 2: Dataset Load Karo

In [ ]:
# MNIST dataset load karo
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print('=' * 45)
print('📁 DATASET INFORMATION')
print('=' * 45)
print(f'Training Images  : {x_train.shape[0]:,}')
print(f'Testing Images   : {x_test.shape[0]:,}')
print(f'Image Size       : {x_train.shape[1]} x {x_train.shape[2]} pixels')
print(f'Pixel Value Range: {x_train.min()} - {x_train.max()}')
print(f'Classes          : {np.unique(y_train)}')
print('=' * 45)

## 🖼️ Step 3: Sample Images Dekho

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle('🔢 MNIST Dataset - Sample Images (0-9)', fontsize=14, fontweight='bold')

for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    axes[0, digit].imshow(x_train[idx], cmap='gray')
    axes[0, digit].set_title(f'Digit: {digit}', fontsize=9)
    axes[0, digit].axis('off')
    
    idx2 = np.where(y_train == digit)[0][1]
    axes[1, digit].imshow(x_train[idx2], cmap='gray')
    axes[1, digit].axis('off')

plt.tight_layout()
plt.savefig('sample_digits.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample images show ho gayi!')

## ⚙️ Step 4: Preprocessing - Data Tayyar Karo

In [ ]:
# 1. Normalize: 0-255 ko 0-1 mein convert karo
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32') / 255.0

# 2. Flatten: 28x28 ko 784 vector mein badlo
x_train_flat = x_train_norm.reshape(-1, 784)
x_test_flat  = x_test_norm.reshape(-1, 784)

# 3. Labels ko One-Hot Encode karo
y_train_ohe = to_categorical(y_train, num_classes=10)
y_test_ohe  = to_categorical(y_test,  num_classes=10)

# 4. Validation set alag karo (training ka 10%)
val_split = int(0.9 * len(x_train_flat))
x_val = x_train_flat[val_split:]
y_val = y_train_ohe[val_split:]
x_tr  = x_train_flat[:val_split]
y_tr  = y_train_ohe[:val_split]

print('✅ Preprocessing Complete!')
print(f'   Training Set   : {x_tr.shape}')
print(f'   Validation Set : {x_val.shape}')
print(f'   Test Set       : {x_test_flat.shape}')
print(f'   Pixel range after normalization: {x_train_norm.min():.1f} - {x_train_norm.max():.1f}')

## ⚡ Step 5: Experiment 1 - Activation Functions Compare Karo

In [ ]:
def build_model_with_activation(activation_name):
    """Different activation functions test karne ke liye"""
    model = Sequential([
        Dense(128, activation=activation_name, input_shape=(784,)),
        Dense(64,  activation=activation_name),
        Dense(10,  activation='softmax')  # Output hamesha softmax
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

activation_results = {}
activations_to_test = ['relu', 'sigmoid', 'tanh']

print('🔬 Activation Functions Test ho rahi hain...')
print('-' * 50)

for act in activations_to_test:
    print(f'  Testing: {act.upper()}...')
    model = build_model_with_activation(act)
    history = model.fit(
        x_tr, y_tr,
        epochs=5,
        batch_size=128,
        validation_data=(x_val, y_val),
        verbose=0
    )
    _, acc = model.evaluate(x_test_flat, y_test_ohe, verbose=0)
    activation_results[act] = {
        'history': history,
        'test_accuracy': acc
    }
    print(f'  ✅ {act.upper()} Test Accuracy: {acc*100:.2f}%')

print('-' * 50)
best_act = max(activation_results, key=lambda x: activation_results[x]['test_accuracy'])
print(f'\n🏆 Best Activation: {best_act.upper()}')

In [ ]:
# Activation Functions ka graph
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = {'relu': '#2196F3', 'sigmoid': '#FF5722', 'tanh': '#4CAF50'}

for act in activations_to_test:
    hist = activation_results[act]['history']
    ax1.plot(hist.history['val_accuracy'], label=act.upper(),
             color=colors[act], linewidth=2, marker='o')
    ax2.plot(hist.history['val_loss'], label=act.upper(),
             color=colors[act], linewidth=2, marker='s')

ax1.set_title('Validation Accuracy - Activation Comparison', fontweight='bold')
ax1.set_xlabel('Epochs'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_title('Validation Loss - Activation Comparison', fontweight='bold')
ax2.set_xlabel('Epochs'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('activation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏗️ Step 6: Final Model - Best Architecture Build Karo

In [ ]:
# Best Model: Dropout ke saath
final_model = Sequential([
    # Input + Hidden Layer 1
    Dense(512, activation='relu', input_shape=(784,),
          kernel_initializer='he_normal'),
    Dropout(0.3),
    
    # Hidden Layer 2
    Dense(256, activation='relu', kernel_initializer='he_normal'),
    Dropout(0.3),
    
    # Hidden Layer 3
    Dense(128, activation='relu', kernel_initializer='he_normal'),
    Dropout(0.2),
    
    # Hidden Layer 4
    Dense(64, activation='relu', kernel_initializer='he_normal'),
    
    # Output Layer
    Dense(10, activation='softmax')
])

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('✅ Final Model Architecture:')
final_model.summary()

## 🚀 Step 7: Model Train Karo

In [ ]:
# Callbacks - Smart Training
callbacks = [
    # Agar improvement na ho toh learning rate kam karo
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Agar zyada improve na ho toh training rok do
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print('🚀 Training shuru ho rahi hai...')
print('=' * 50)

history = final_model.fit(
    x_tr, y_tr,
    epochs=30,
    batch_size=64,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Training Complete!')

## 📈 Step 8: Training Graphs - Accuracy & Loss

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = len(history.history['accuracy'])
epoch_range = range(1, epochs_ran + 1)

# Accuracy Graph
ax1.plot(epoch_range, history.history['accuracy'],     'b-', linewidth=2, label='Training Accuracy')
ax1.plot(epoch_range, history.history['val_accuracy'], 'r--', linewidth=2, label='Validation Accuracy')
ax1.fill_between(epoch_range,
                 history.history['accuracy'],
                 history.history['val_accuracy'],
                 alpha=0.1, color='purple')
ax1.set_title('📈 Model Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=11); ax1.grid(True, alpha=0.3)
ax1.set_ylim([0.9, 1.0])

# Loss Graph
ax2.plot(epoch_range, history.history['loss'],     'b-', linewidth=2, label='Training Loss')
ax2.plot(epoch_range, history.history['val_loss'], 'r--', linewidth=2, label='Validation Loss')
ax2.set_title('📉 Model Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(fontsize=11); ax2.grid(True, alpha=0.3)

plt.suptitle('Training vs Validation Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎯 Step 9: Model Evaluate Karo - Final Accuracy

In [ ]:
test_loss, test_accuracy = final_model.evaluate(x_test_flat, y_test_ohe, verbose=0)

print('=' * 50)
print('🎯 FINAL MODEL RESULTS')
print('=' * 50)
print(f'  Test Loss     : {test_loss:.4f}')
print(f'  Test Accuracy : {test_accuracy*100:.2f}%')
print(f'  Error Rate    : {(1-test_accuracy)*100:.2f}%')
print('=' * 50)

if test_accuracy >= 0.98:
    print('🏆 EXCELLENT! 98%+ Accuracy Achieved!')
elif test_accuracy >= 0.97:
    print('✅ GREAT! 97%+ Accuracy Achieved!')
else:
    print('👍 Good start! Try more epochs.')

## 🗺️ Step 10: Confusion Matrix

In [ ]:
y_pred_prob = final_model.predict(x_test_flat, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = y_test

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5, ax=ax)
ax.set_title('🗺️ Confusion Matrix\n(Rows=Actual, Columns=Predicted)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Classification Report:')
print(classification_report(y_true, y_pred, target_names=[str(i) for i in range(10)]))

## 🔍 Step 11: Misclassified Examples Dekho

In [ ]:
# Galat predictions dhundho
wrong_idx = np.where(y_pred != y_true)[0]
print(f'Total Misclassified: {len(wrong_idx)} out of {len(y_true)}')
print(f'Accuracy: {(1 - len(wrong_idx)/len(y_true))*100:.2f}%')

# 10 galat examples show karo
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('❌ Misclassified Examples (Model ne galat predict kiya)', fontsize=13, fontweight='bold')

for i, idx in enumerate(wrong_idx[:10]):
    row, col = divmod(i, 5)
    confidence = y_pred_prob[idx][y_pred[idx]] * 100
    axes[row, col].imshow(x_test[idx], cmap='Reds', alpha=0.8)
    axes[row, col].set_title(
        f'True: {y_true[idx]}  |  Pred: {y_pred[idx]}\nConfidence: {confidence:.1f}%',
        fontsize=9, color='darkred'
    )
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('misclassified.png', dpi=150, bbox_inches='tight')
plt.show()

## 🆚 Step 12: Overfitting Analysis - Dropout ke saath aur baghair

In [ ]:
# Model WITHOUT Dropout
model_no_dropout = Sequential([
    Dense(512, activation='relu', input_shape=(784,)),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(10,  activation='softmax')
])
model_no_dropout.compile(optimizer='adam',
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])

print('Training WITHOUT Dropout...')
hist_no_drop = model_no_dropout.fit(
    x_tr, y_tr, epochs=15, batch_size=64,
    validation_data=(x_val, y_val), verbose=0
)

# Model WITH Dropout
model_with_dropout = Sequential([
    Dense(512, activation='relu', input_shape=(784,)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(10,  activation='softmax')
])
model_with_dropout.compile(optimizer='adam',
                           loss='categorical_crossentropy',
                           metrics=['accuracy'])

print('Training WITH Dropout...')
hist_drop = model_with_dropout.fit(
    x_tr, y_tr, epochs=15, batch_size=64,
    validation_data=(x_val, y_val), verbose=0
)
print('✅ Done!')

In [ ]:
# Overfitting Comparison Graph
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
e = range(1, 16)

# WITHOUT Dropout
axes[0].plot(e, hist_no_drop.history['accuracy'],     'b-',  linewidth=2, label='Train')
axes[0].plot(e, hist_no_drop.history['val_accuracy'], 'r--', linewidth=2, label='Validation')
gap_no = np.array(hist_no_drop.history['accuracy']) - np.array(hist_no_drop.history['val_accuracy'])
axes[0].fill_between(e, hist_no_drop.history['accuracy'],
                        hist_no_drop.history['val_accuracy'],
                        alpha=0.2, color='red', label=f'Overfit Gap (max={gap_no.max():.3f})')
axes[0].set_title('❌ WITHOUT Dropout (Overfitting!)', fontsize=12, fontweight='bold', color='red')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim([0.9, 1.0])

# WITH Dropout
axes[1].plot(e, hist_drop.history['accuracy'],     'b-',  linewidth=2, label='Train')
axes[1].plot(e, hist_drop.history['val_accuracy'], 'r--', linewidth=2, label='Validation')
gap_d = np.array(hist_drop.history['accuracy']) - np.array(hist_drop.history['val_accuracy'])
axes[1].fill_between(e, hist_drop.history['accuracy'],
                       hist_drop.history['val_accuracy'],
                       alpha=0.2, color='green', label=f'Gap (max={gap_d.max():.3f})')
axes[1].set_title('✅ WITH Dropout (Better Generalization!)', fontsize=12, fontweight='bold', color='green')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim([0.9, 1.0])

plt.suptitle('🔬 Overfitting Analysis: Dropout Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('overfitting_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎮 Step 13: Live Predictions - Model Test Karo

In [ ]:
# Random test images pe prediction karo
n_samples = 12
random_idx = np.random.choice(len(x_test), n_samples, replace=False)

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle('🎯 Live Predictions on Random Test Images', fontsize=14, fontweight='bold')

for i, idx in enumerate(random_idx):
    row, col = divmod(i, 6)
    img_flat    = x_test_flat[idx:idx+1]
    pred_probs  = final_model.predict(img_flat, verbose=0)[0]
    pred_label  = np.argmax(pred_probs)
    true_label  = y_test[idx]
    confidence  = pred_probs[pred_label] * 100
    is_correct  = pred_label == true_label

    axes[row, col].imshow(x_test[idx], cmap='gray')
    color = 'green' if is_correct else 'red'
    icon  = '✅' if is_correct else '❌'
    axes[row, col].set_title(
        f'{icon} True:{true_label}  Pred:{pred_label}\n{confidence:.1f}%',
        fontsize=9, color=color, fontweight='bold'
    )
    for spine in axes[row, col].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
    axes[row, col].set_xticks([])
    axes[row, col].set_yticks([])

plt.tight_layout()
plt.savefig('live_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Step 14: Experiment 2 - Hidden Layers Compare Karo

In [ ]:
def build_model_n_layers(n_layers):
    layers_list = [Dense(128, activation='relu', input_shape=(784,))]
    for _ in range(n_layers - 1):
        layers_list.append(Dense(64, activation='relu'))
    layers_list.append(Dense(10, activation='softmax'))
    model = Sequential(layers_list)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

layer_results = {}
print('🔬 Hidden Layers Experiment...')

for n in [1, 2, 3, 4]:
    m = build_model_n_layers(n)
    h = m.fit(x_tr, y_tr, epochs=5, batch_size=128,
              validation_data=(x_val, y_val), verbose=0)
    _, acc = m.evaluate(x_test_flat, y_test_ohe, verbose=0)
    layer_results[n] = acc
    print(f'  {n} Hidden Layer(s): Test Accuracy = {acc*100:.2f}%')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([f'{n} Layer(s)' for n in layer_results.keys()],
              [v*100 for v in layer_results.values()],
              color=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4'],
              edgecolor='black', linewidth=1.2)
for bar, val in zip(bars, layer_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            f'{val*100:.2f}%', ha='center', va='bottom', fontweight='bold')
ax.set_title('Hidden Layers vs Test Accuracy', fontsize=13, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)')
ax.set_ylim([95, 100])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('layers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏁 Step 15: Final Summary Report

In [ ]:
final_loss, final_acc = final_model.evaluate(x_test_flat, y_test_ohe, verbose=0)

print('=' * 55)
print('       📋 COMPLETE PROJECT SUMMARY REPORT')
print('=' * 55)
print(f'  Dataset          : MNIST (70,000 images)')
print(f'  Model Type       : Dense Neural Network (MLP)')
print(f'  Architecture     : 784 → 512 → 256 → 128 → 64 → 10')
print(f'  Activation       : ReLU (hidden) + Softmax (output)')
print(f'  Optimizer        : Adam')
print(f'  Regularization   : Dropout (0.2-0.3)')
print(f'  Callbacks        : EarlyStopping + ReduceLROnPlateau')
print('-' * 55)
print(f'  ✅ Final Test Accuracy : {final_acc*100:.2f}%')
print(f'  ✅ Final Test Loss     : {final_loss:.4f}')
print(f'  ✅ Error Rate          : {(1-final_acc)*100:.2f}%')
print(f'  ✅ Correct Predictions : {int(final_acc * 10000):,} / 10,000')
print('=' * 55)
print()
print('  📁 Saved Files:')
print('     • sample_digits.png')
print('     • activation_comparison.png')
print('     • training_history.png')
print('     • confusion_matrix.png')
print('     • misclassified.png')
print('     • overfitting_comparison.png')
print('     • live_predictions.png')
print('     • layers_comparison.png')
print('=' * 55)
print('  🎓 Project Successfully Complete!')
print('=' * 55)